# NSW Active-Fire Reliability Pilot

> **Important Interpretation Callout:** This notebook performs a spatiotemporal reliability audit matching satellite hotspot observations (DEA Hotspots) against official post-event fire boundary records (NPWS Fire History). This is a *calibration and reliability study* of spatial-temporal overlap, **not a detector-accuracy evaluation**. Unmatched observations are labelled as *unresolved*, and must not be assumed to be errors or sensor inaccuracies without independent ground truth.

In [1]:
# Execution Configuration
EXECUTION_MODE = "snapshot"  # Options: "snapshot", "live_refresh"
SNAPSHOT_SLUG = "tuannm3812/nsw-active-fire-pilot-snapshot"

import json
import hashlib
import os
from pathlib import Path
from datetime import datetime, timedelta, timezone
import math
from typing import Dict, List, Tuple, Optional, Sequence
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


## 1. Project overview

This project evaluates the spatiotemporal reliability of satellite active-fire hotspots in New South Wales (NSW), Australia. Satellite-derived hotspot products (such as those from MODIS, VIIRS, and AHI) are widely used for real-time fire detection, but their operational reliability must be calibrated against official historical fire boundaries to understand spatial and temporal alignment. We focus on a bounded region of NSW during a fortnight of intense fire activity in January 2020.

## 2. Data and methodology

We utilize two primary public datasets:
1. **DEA Hotspots WFS:** Historical active-fire point observations collected by Geoscience Australia, including attributes like satellite, sensor, acquisition time, temp_kelvin, power, confidence, and positional accuracy.
2. **NPWS Fire History Layer:** Polygon boundaries representing wildfires and prescribed burns managed by the NSW National Parks and Wildlife Service (NPWS), explicitly licensed under Creative Commons Attribution (CC BY 4.0).

**Methodology:**
- **Exact Matching:** A hotspot point is matched if it falls directly inside a fire boundary polygon, and its observation time is within the fire's ignition-to-extinguish window (plus a symmetric 1-day temporal grace period).
- **Sensor-Buffered Matching:** The match criteria are expanded by buffering the fire polygons using each sensor's documented positional accuracy (e.g., ±0.375 km for VIIRS, ±1 km for MODIS).

In [2]:
from datetime import datetime, timedelta, timezone
import math
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


Point = Tuple[float, float]


def _coordinate_points(coordinates):
    if not coordinates:
        return
    if isinstance(coordinates[0], (int, float)):
        yield float(coordinates[0]), float(coordinates[1])
        return
    for part in coordinates:
        yield from _coordinate_points(part)


def geometry_bounds(geometry: Dict) -> Tuple[float, float, float, float]:
    points = list(_coordinate_points(geometry.get("coordinates", [])))
    if not points:
        raise ValueError("Geometry has no coordinates")
    longitudes = [point[0] for point in points]
    latitudes = [point[1] for point in points]
    return min(longitudes), min(latitudes), max(longitudes), max(latitudes)


def prepare_features(features: Iterable[Dict]) -> List[Dict]:
    prepared = []
    for feature in features:
        copy = dict(feature)
        copy["_bbox"] = geometry_bounds(copy["geometry"])
        prepared.append(copy)
    return prepared


def _point_in_bbox(lon: float, lat: float, bbox: Tuple[float, float, float, float]) -> bool:
    return bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]


def _point_segment_distance_km(point: Point, start: Point, end: Point) -> float:
    lon_scale = 111.320 * math.cos(math.radians(point[1]))
    lat_scale = 110.574
    ax = (start[0] - point[0]) * lon_scale
    ay = (start[1] - point[1]) * lat_scale
    bx = (end[0] - point[0]) * lon_scale
    by = (end[1] - point[1]) * lat_scale
    segment_x = bx - ax
    segment_y = by - ay
    denominator = segment_x * segment_x + segment_y * segment_y
    if denominator == 0:
        return math.hypot(ax, ay)
    projection = max(0.0, min(1.0, -(ax * segment_x + ay * segment_y) / denominator))
    return math.hypot(ax + projection * segment_x, ay + projection * segment_y)


def point_within_geometry_buffer(
    lon: float, lat: float, geometry: Dict, buffer_km: float
) -> bool:
    if point_in_geometry(lon, lat, geometry):
        return True
    point = (float(lon), float(lat))
    minimum = float("inf")
    coordinates = geometry.get("coordinates", [])
    polygons = [coordinates] if geometry.get("type") == "Polygon" else coordinates
    for polygon in polygons:
        for ring in polygon:
            for index in range(len(ring)):
                start = (float(ring[index - 1][0]), float(ring[index - 1][1]))
                end = (float(ring[index][0]), float(ring[index][1]))
                minimum = min(minimum, _point_segment_distance_km(point, start, end))
                if minimum <= buffer_km:
                    return True
    return False


def _point_on_segment(point: Point, start: Point, end: Point, tolerance: float = 1e-10) -> bool:
    px, py = point
    x1, y1 = start
    x2, y2 = end
    cross = (px - x1) * (y2 - y1) - (py - y1) * (x2 - x1)
    if abs(cross) > tolerance:
        return False
    return (
        min(x1, x2) - tolerance <= px <= max(x1, x2) + tolerance
        and min(y1, y2) - tolerance <= py <= max(y1, y2) + tolerance
    )


def _point_in_ring(point: Point, ring: Sequence[Sequence[float]]) -> bool:
    inside = False
    for index in range(len(ring)):
        start = (float(ring[index - 1][0]), float(ring[index - 1][1]))
        end = (float(ring[index][0]), float(ring[index][1]))
        if _point_on_segment(point, start, end):
            return True
        x1, y1 = start
        x2, y2 = end
        crosses = (y1 > point[1]) != (y2 > point[1])
        if crosses:
            intersection_x = (x2 - x1) * (point[1] - y1) / (y2 - y1) + x1
            if point[0] < intersection_x:
                inside = not inside
    return inside


def _point_in_polygon(point: Point, polygon: Sequence[Sequence[Sequence[float]]]) -> bool:
    if not polygon or not _point_in_ring(point, polygon[0]):
        return False
    return not any(_point_in_ring(point, hole) for hole in polygon[1:])


def point_in_geometry(lon: float, lat: float, geometry: Dict) -> bool:
    """Return whether a WGS84 point lies in a GeoJSON Polygon or MultiPolygon."""
    point = (float(lon), float(lat))
    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates", [])
    if geometry_type == "Polygon":
        return _point_in_polygon(point, coordinates)
    if geometry_type == "MultiPolygon":
        return any(_point_in_polygon(point, polygon) for polygon in coordinates)
    raise ValueError("Only Polygon and MultiPolygon geometries are supported")


def parse_datetime(value) -> Optional[datetime]:
    if value in (None, ""):
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value / 1000, tz=timezone.utc)
    text = str(value).strip().replace("Z", "+00:00")
    parsed = datetime.fromisoformat(text)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def within_event_window(observed_at: datetime, properties: Dict, grace_days: int) -> bool:
    """Check an observation against ignition/extinguish dates plus symmetric grace."""
    ignition = parse_datetime(properties.get("ignition_date"))
    if ignition is None:
        return False
    extinguish = parse_datetime(properties.get("extinguish_date")) or ignition
    grace = timedelta(days=grace_days)
    observed = parse_datetime(observed_at)
    return ignition - grace <= observed <= extinguish + grace


def _normalized_fire_type(value: Optional[str]) -> str:
    normalized = (value or "").strip().lower().replace(" ", "_")
    if normalized in {"bushfire", "wildfire", "wild_fire"}:
        return "bushfire"
    if normalized in {"prescribed_burn", "hazard_reduction", "planned_burn"}:
        return "prescribed_burn"
    return "other_fire"


def classify_hotspot(
    hotspot: Dict,
    features: Iterable[Dict],
    grace_days: int = 1,
    spatial_buffer_km: float = 0.0,
) -> Dict:
    """Classify one hotspot using polygon containment and the event date window."""
    observed_at = parse_datetime(hotspot["datetime"])
    spatial_matches: List[Dict] = []
    temporal_matches: List[Dict] = []
    for feature in features:
        if feature.get("_bbox"):
            degree_buffer = spatial_buffer_km / 80.0
            bbox = feature["_bbox"]
            expanded = (
                bbox[0] - degree_buffer,
                bbox[1] - degree_buffer,
                bbox[2] + degree_buffer,
                bbox[3] + degree_buffer,
            )
            if not _point_in_bbox(
                float(hotspot["longitude"]), float(hotspot["latitude"]), expanded
            ):
                continue
        if point_within_geometry_buffer(
            hotspot["longitude"],
            hotspot["latitude"],
            feature["geometry"],
            spatial_buffer_km,
        ):
            spatial_matches.append(feature)
            if within_event_window(observed_at, feature.get("properties", {}), grace_days):
                temporal_matches.append(feature)

    if temporal_matches:
        chosen = sorted(
            temporal_matches,
            key=lambda feature: (
                parse_datetime(feature.get("properties", {}).get("ignition_date"))
                or datetime.max.replace(tzinfo=timezone.utc),
                str(feature.get("properties", {}).get("fire_id") or ""),
            ),
        )[0]
        properties = chosen.get("properties", {})
        return {
            **hotspot,
            "match_class": _normalized_fire_type(properties.get("fire_type")),
            "fire_id": properties.get("fire_id"),
            "fire_name": properties.get("fire_name"),
            "spatial_match_count": len(spatial_matches),
            "temporal_match_count": len(temporal_matches),
        }

    return {
        **hotspot,
        "match_class": "spatial_only" if spatial_matches else "unmatched",
        "fire_id": None,
        "fire_name": None,
        "spatial_match_count": len(spatial_matches),
        "temporal_match_count": 0,
    }

In [3]:
import pandas as pd
import numpy as np


def headline_summary(exact: pd.DataFrame, buffered: pd.DataFrame) -> dict:
    total_exact = len(exact)
    total_buffered = len(buffered)
    if total_exact != total_buffered:
        raise ValueError("Exact and buffered dataframes must have the same length")
        
    exact_matches = sum(exact["match_class"] != "unmatched")
    exact_unresolved = total_exact - exact_matches
    
    buffered_matches = sum(buffered["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"]))
    buffered_unresolved = total_buffered - buffered_matches
    
    return {
        "total_hotspots": total_exact,
        "exact_matches": int(exact_matches),
        "exact_unresolved": int(exact_unresolved),
        "exact_match_rate": float(exact_matches / total_exact) if total_exact > 0 else 0.0,
        "buffered_matches": int(buffered_matches),
        "buffered_unresolved": int(buffered_unresolved),
        "buffered_match_rate": float(buffered_matches / total_buffered) if total_buffered > 0 else 0.0,
    }


def sensor_summary(matches: pd.DataFrame) -> pd.DataFrame:
    summary_list = []
    grouped = matches.groupby("sensor")
    for sensor, group in grouped:
        total = len(group)
        matched = sum(group["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"]))
        unresolved = total - matched
        summary_list.append({
            "sensor": sensor,
            "total_hotspots": total,
            "matched_hotspots": int(matched),
            "unresolved_hotspots": int(unresolved),
            "match_rate": float(matched / total) if total > 0 else 0.0
        })
    return pd.DataFrame(summary_list)


def event_concentration(matches: pd.DataFrame) -> pd.DataFrame:
    # Filter for matched hotspots only
    matched = matches[matches["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"])]
    if matched.empty:
        return pd.DataFrame(columns=["fire_name", "fire_id", "matched_hotspots", "percentage"])
        
    total_matched = len(matched)
    # Group by fire_name and fire_id (or OBJECTID)
    grouped = matched.groupby(["fire_name", "fire_id"], dropna=False)
    summary_list = []
    for (fire_name, fire_id), group in grouped:
        count = len(group)
        summary_list.append({
            "fire_name": fire_name if pd.notna(fire_name) else "Unknown",
            "fire_id": fire_id if pd.notna(fire_id) else "Unknown",
            "matched_hotspots": count,
            "percentage": float(count / total_matched) if total_matched > 0 else 0.0
        })
    df = pd.DataFrame(summary_list)
    return df.sort_values(by="matched_hotspots", ascending=False).reset_index(drop=True)


def deterministic_display_sample(frame: pd.DataFrame, size: int, seed: int = 42) -> pd.DataFrame:
    n = min(len(frame), size)
    if n <= 0:
        return frame.copy()
    return frame.sample(n=n, random_state=seed).sort_index()


def compare_refresh(reviewed: dict, refreshed: dict) -> pd.DataFrame:
    comparison_list = []
    for key in reviewed:
        rev_val = reviewed[key]
        ref_val = refreshed.get(key)
        status = "stable" if rev_val == ref_val else "changed"
        comparison_list.append({
            "metric": key,
            "reviewed": rev_val,
            "refreshed": ref_val,
            "status": status
        })
    return pd.DataFrame(comparison_list)


def assert_snapshot_invariants(actual: dict, expected: dict) -> None:
    differences = {key: (expected[key], actual.get(key)) for key in expected if actual.get(key) != expected[key]}
    if differences:
        raise AssertionError(f"Snapshot invariants changed: {differences}")


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import Dict, List

OKABE_ITO_COLORS = ["#0072B2", "#E69F00", "#CC79A7", "#999999"]


def plot_sensor_composition(frame: pd.DataFrame) -> plt.Figure:
    """Plot the total count of hotspots by sensor."""
    fig, ax = plt.subplots(figsize=(6, 4.5), facecolor="white")
    ax.set_facecolor("white")
    
    # Sort for consistent display
    df = frame.sort_values(by="total_hotspots", ascending=True)
    total_n = df["total_hotspots"].sum()
    
    bars = ax.barh(df["sensor"], df["total_hotspots"], color=OKABE_ITO_COLORS[0], edgecolor="none")
    ax.set_xlabel("Total Hotspot Observations", color="black", fontsize=10)
    ax.set_title(f"Hotspot Observations by Sensor (Total N={total_n:,})", color="black", fontsize=12, pad=15)
    
    # Style axes
    ax.tick_params(colors="black", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + (width * 0.01) + 1, bar.get_y() + bar.get_height()/2, f"{int(width):,}", 
                va="center", ha="left", color="black", fontsize=9)
                
    fig.tight_layout()
    return fig


def plot_match_rates(frame: pd.DataFrame) -> plt.Figure:
    """Plot match rate by sensor with denominators in the labels."""
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor="white")
    ax.set_facecolor("white")
    
    # Create labels with denominators
    labels = []
    rates = []
    for _, row in frame.iterrows():
        sensor = row["sensor"]
        n = int(row["total_hotspots"])
        rate = float(row["match_rate"])
        labels.append(f"{sensor}\n(n={n:,})")
        rates.append(rate * 100.0)  # Percentage
        
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, rates, color=OKABE_ITO_COLORS[1], edgecolor="none")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, color="black", fontsize=9)
    ax.set_xlabel("Spatiotemporal Match Rate (%)", color="black", fontsize=10)
    ax.set_title("Active-Fire Match Rate by Sensor", color="black", fontsize=12, pad=15)
    ax.set_xlim(0, 105)
    
    # Style axes
    ax.tick_params(colors="black", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f"{width:.1f}%", 
                va="center", ha="left", color="black", fontsize=9)
                
    fig.tight_layout()
    return fig


def plot_confidence_by_algorithm(frame: pd.DataFrame) -> plt.Figure:
    """Plot confidence distributions by sensor / algorithm."""
    fig, ax = plt.subplots(figsize=(7, 5.0), facecolor="white")
    ax.set_facecolor("white")
    
    # Prepare data for boxplot
    sensors = sorted(frame["sensor"].dropna().unique())
    data = []
    labels = []
    for sensor in sensors:
        subset = frame[frame["sensor"] == sensor]["confidence"].dropna()
        if not subset.empty:
            data.append(subset.values)
            labels.append(f"{sensor}\n(n={len(subset):,})")
            
    if data:
        # Custom boxplot styling with Matplotlib 3.9+ compatibility
        try:
            import re
            v_parts = [int(x) for x in re.findall(r"\d+", matplotlib.__version__)]
        except Exception:
            v_parts = [3, 0]
        use_tick_labels = len(v_parts) >= 2 and (v_parts[0] > 3 or (v_parts[0] == 3 and v_parts[1] >= 9))
        
        if use_tick_labels:
            box = ax.boxplot(data, tick_labels=labels, patch_artist=True, medianprops={"color": "black", "linewidth": 1.5})
        else:
            box = ax.boxplot(data, labels=labels, patch_artist=True, medianprops={"color": "black", "linewidth": 1.5})
            
        for patch in box["boxes"]:
            patch.set_facecolor(OKABE_ITO_COLORS[2])
            patch.set_edgecolor("black")
            
    ax.set_ylabel("Confidence Score / Value", color="black", fontsize=10)
    ax.set_title("Hotspot Confidence Distribution by Sensor", color="black", fontsize=12, pad=15)
    
    ax.tick_params(colors="black", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    
    fig.tight_layout()
    return fig


def plot_event_concentration(frame: pd.DataFrame) -> plt.Figure:
    """Plot hotspot count by fire event to show spatial concentration."""
    fig, ax = plt.subplots(figsize=(7, 5.0), facecolor="white")
    ax.set_facecolor("white")
    
    # Display top 10 events for readability
    top_events = frame.head(10).copy()
    total_matched = frame["matched_hotspots"].sum()
    
    # Shorten long event names
    labels = []
    for _, row in top_events.iterrows():
        name = str(row["fire_name"])
        if len(name) > 20:
            name = name[:18] + "..."
        labels.append(name)
        
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, top_events["matched_hotspots"], color=OKABE_ITO_COLORS[0], edgecolor="none")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, color="black", fontsize=9)
    ax.invert_yaxis()  # top-down
    ax.set_xlabel("Number of Matched Hotspots", color="black", fontsize=10)
    ax.set_title(f"Concentration of Matches Across Fire Events (Total Matched N={total_matched:,})", color="black", fontsize=12, pad=15)
    
    ax.tick_params(colors="black", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + (width * 0.01) + 1, bar.get_y() + bar.get_height()/2, f"{int(width):,}", 
                va="center", ha="left", color="black", fontsize=9)
                
    fig.tight_layout()
    return fig


def plot_pilot_map(hotspots: pd.DataFrame, polygons: List[dict], displayed_n: int) -> plt.Figure:
    """Plot map showing hotspots and fire event boundary polygons."""
    fig, ax = plt.subplots(figsize=(8, 7.0), facecolor="white")
    ax.set_facecolor("white")
    
    # 1. Plot polygons
    for feature in polygons:
        geometry = feature.get("geometry", {})
        coords = geometry.get("coordinates", [])
        geom_type = geometry.get("type")
        
        poly_list = [coords] if geom_type == "Polygon" else coords
        for poly in poly_list:
            for ring in poly:
                x = [pt[0] for pt in ring]
                y = [pt[1] for pt in ring]
                ax.plot(x, y, color="black", linewidth=1.2, alpha=0.8, zorder=2)
                ax.fill(x, y, color="#999999", alpha=0.15, zorder=1)
                
    # 2. Plot hotspots
    # Filter or sample hotspots
    matched = hotspots[hotspots["match_class"].isin(["bushfire", "prescribed_burn", "other_fire", "spatial_only"])]
    unmatched = hotspots[hotspots["match_class"] == "unmatched"]
    
    # Plot matched
    ax.scatter(matched["longitude"], matched["latitude"], color=OKABE_ITO_COLORS[1], 
               label="Matched Hotspot", s=12, alpha=0.7, zorder=4, marker="o", edgecolors="none")
               
    # Plot unmatched
    ax.scatter(unmatched["longitude"], unmatched["latitude"], color=OKABE_ITO_COLORS[0], 
               label="Unresolved Hotspot", s=8, alpha=0.5, zorder=3, marker="x")
               
    ax.set_xlabel("Longitude (WGS84)", color="black", fontsize=10)
    ax.set_ylabel("Latitude (WGS84)", color="black", fontsize=10)
    ax.set_title(f"Active-Fire Spatial Distribution (Sample Size: {displayed_n:,})", color="black", fontsize=12, pad=15)
    
    ax.legend(facecolor="white", edgecolor="black", labelcolor="black", loc="upper right")
    ax.tick_params(colors="black", labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.3, color="#999999")
    
    fig.tight_layout()
    return fig


## 3. Results

We load the datasets from the snapshot package and perform the spatiotemporal matching. We compare the match rates and unresolved observations between the exact matching baseline and the sensor-buffered matching pipeline.

In [5]:
import re

def normalize_hotspot(feature: dict) -> dict:
    properties = feature.get('properties', {})
    coordinates = feature.get('geometry', {}).get('coordinates', [None, None])
    return {
        'id': properties.get('id'),
        'datetime': properties.get('datetime'),
        'longitude': properties.get('longitude', coordinates[0]),
        'latitude': properties.get('latitude', coordinates[1]),
        'sensor': properties.get('sensor'),
        'satellite': properties.get('satellite'),
        'process_algorithm': properties.get('process_algorithm'),
        'confidence': properties.get('confidence'),
        'accuracy': properties.get('accuracy'),
    }

def parse_accuracy_km(value) -> float:
    if value in (None, ''):
        return 0.0
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)', str(value))
    return float(match.group(1)) if match else 0.0

# Helper to map NPWS schema to standard RFS-like fields consumed by matching code
def map_npws_to_rfs(feature: dict) -> dict:
    props = feature.get('properties', {})
    mapped_props = {
        'fire_id': props.get('FireNo') or str(props.get('OBJECTID')),
        'fire_name': props.get('FireName') or 'Unnamed NPWS Event',
        'ignition_date': props.get('StartDate'),
        'extinguish_date': props.get('EndDate'),
        'fire_type': 'bushfire' if props.get('FireType') == 1 else 'prescribed_burn',
        'area_ha': props.get('AreaHa'),
        'perim_km': (props.get('PerimeterM') or 0.0) / 1000.0,
        'state': 'NSW',
        'agency': 'NPWS'
    }
    return {
        'type': feature.get('type'),
        'properties': mapped_props,
        'geometry': feature.get('geometry')
    }

# Define file paths based on execution mode
if EXECUTION_MODE == 'snapshot':
    import os
    # Check Kaggle input dataset, recursive search, or local fallback
    dataset_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'dea_hotspots.geojson' in files:
            dataset_dir = Path(root)
            print(f"Found dataset at: {dataset_dir}")
            break
    if dataset_dir is None:
        for root, dirs, files in os.walk('../input'):
            if 'dea_hotspots.geojson' in files:
                dataset_dir = Path(root)
                print(f"Found dataset at: {dataset_dir}")
                break
    if dataset_dir is None:
        dataset_dir = Path('.')
        print(f"Dataset not found in Kaggle inputs. Falling back to local: {dataset_dir}")
    dea_path = dataset_dir / 'dea_hotspots.geojson'
    npws_path = dataset_dir / 'npws_fire_history.geojson'
else:
    # Live refresh (download fresh features)
    raise NotImplementedError('Live refresh from ArcGIS/WFS endpoints requires direct network access')

# Load datasets
print('Loading spatial datasets...')
with open(dea_path) as f:
    dea_data = json.load(f)
with open(npws_path) as f:
    npws_data = json.load(f)

# Map and prepare features
npws_mapped = [map_npws_to_rfs(f) for f in npws_data['features']]
features = prepare_features(npws_mapped)
hotspots = [normalize_hotspot(h) for h in dea_data['features']]

# Perform exact spatiotemporal matching
print('Running exact matching...')
exact_matches = [classify_hotspot(h, features, grace_days=1, spatial_buffer_km=0.0) for h in hotspots]
df_exact = pd.DataFrame(exact_matches)

# Perform sensor-buffered matching
print('Running sensor-buffered matching...')
buffered_matches = [classify_hotspot(h, features, grace_days=1, spatial_buffer_km=parse_accuracy_km(h.get('accuracy'))) for h in hotspots]
df_buffered = pd.DataFrame(buffered_matches)

# Compute headline summary metrics
headline = headline_summary(df_exact, df_buffered)
headline['fire_event_count'] = len(features)
print('\n=== HEADLINE METRICS ===')
for k, v in headline.items():
    if 'rate' in k:
        print(f'{k}: {v * 100.0:.2f}%')
    else:
        print(f'{k}: {v:,}')

# Assert snapshot invariants if in snapshot mode
expected_invariants = {
    'total_hotspots': 19849,
    'fire_event_count': 14,
    'exact_matches': 15334,
    'buffered_matches': 19277,
    'buffered_unresolved': 572,
}
if EXECUTION_MODE == 'snapshot':
    assert_snapshot_invariants(headline, expected_invariants)
    print('\n[SUCCESS] Snapshot invariants verified successfully.')


Dataset not found in Kaggle inputs. Falling back to local: .
Loading spatial datasets...
Running exact matching...


Running sensor-buffered matching...



=== HEADLINE METRICS ===
total_hotspots: 19,849
exact_matches: 15,334
exact_unresolved: 4,515
exact_match_rate: 77.25%
buffered_matches: 19,277
buffered_unresolved: 572
buffered_match_rate: 97.12%
fire_event_count: 14

[SUCCESS] Snapshot invariants verified successfully.


In [6]:
# 1. Generate and display sensor composition plot
fig1 = plot_sensor_composition(pd.DataFrame([
    {'sensor': s, 'total_hotspots': count}
    for s, count in df_exact.groupby('sensor').size().items()
]))
plt.show()

# 2. Generate and display match rates plot
df_sensor = sensor_summary(df_buffered)
fig2 = plot_match_rates(df_sensor)
plt.show()

# 3. Generate confidence distribution boxplot
fig3 = plot_confidence_by_algorithm(df_buffered)
plt.show()

# 4. Generate fire event concentration plot
df_concentration = event_concentration(df_buffered)
fig4 = plot_event_concentration(df_concentration)
plt.show()

# 5. Generate and display spatial distribution map
fig5 = plot_pilot_map(df_buffered, npws_mapped, displayed_n=len(df_buffered))
plt.show()


/var/folders/jm/6w09k7lx7hd5_zrpl1zd8yk40000gn/T/ipykernel_52264/3860079028.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jm/6w09k7lx7hd5_zrpl1zd8yk40000gn/T/ipykernel_52264/3860079028.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jm/6w09k7lx7hd5_zrpl1zd8yk40000gn/T/ipykernel_52264/3860079028.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jm/6w09k7lx7hd5_zrpl1zd8yk40000gn/T/ipykernel_52264/3860079028.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/jm/6w09k7lx7hd5_zrpl1zd8yk40000gn/T/ipykernel_52264/3860079028.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Reliability analysis

Based on the results, we observe that the exact spatiotemporal match rate of active-fire observations against the NPWS Fire History is **77.25%** (15,334 of 19,849 hotspots matched). When taking the sensors' spatial positional accuracy into account via buffering, the match rate increases to **97.12%** (19,277 of 19,849 hotspots matched).

Crucially, the remaining **2.88%** (572 hotspots) are labeled as **unresolved**. These observations represent spatiotemporal offsets that could be caused by:
1. Positional or temporal drift in the satellite products.
2. Off-reserve fire events not captured in the NPWS-specific reserve dataset.
3. Small agricultural burns or brief fire events that did not form a mapped boundary.

This confirms the necessity of using the term *unresolved* rather than assuming they represent errors or omissions, as they may indicate real fire activity outside of NPWS-managed boundaries.

## 5. Research implications

For downstream modeling, this pilot illustrates that spatial and temporal tolerances are critical. A strict containment check (exact matching) under-reports the alignment of satellite hotspots with actual events. By buffering the polygons, we align the official data with the sensors' physical limits, resolving the spatiotemporal offsets. Future research must account for administrative boundaries when analyzing active fires to ensure off-reserve observations are not mischaracterized as errors.

## 6. Reproducibility

To reproduce this analysis, check the local configuration and confirmed data sources:

### Confirmed Public-Source Attributions:
- **DEA Hotspots WFS:** Provided by Geoscience Australia under CC BY 4.0. URL: [DEA Hotspots Service](https://hotspots.dea.ga.gov.au/)
- **NPWS Fire History:** Provided by NSW National Parks and Wildlife Service under Creative Commons Attribution. URL: [Data.NSW NPWS Record](https://data.nsw.gov.au/data/dataset/npws-fire-history)

### Snapshot Provenance Checksums:
- `dea_hotspots.geojson` (SHA-256): `e3fef8c1c9b4a81b07482eca2209885dc0b9c5f08fc5c6ddf310ad39313655d3`
- `npws_fire_history.geojson` (SHA-256): `990507571b2b028c0d5687a7ba4351adb0b7e60ced1a53eddf9eb30e91f92dd5`

In [7]:
# Live Refresh Mode (guarded)
if EXECUTION_MODE == 'live_refresh':
    # In live refresh mode, we would query the active REST services.
    # Since this is a CPU-only private Kaggle runtime with internet disabled by default,
    # any live refresh must be triggered in an authorized environment.
    print('Initializing live refresh...')
    # Rerun code goes here...
